# DWS-Bench: Kaggle benchmark generation and evaluation

This notebook is configured for a Kaggle GPU notebook. It:

1. Locates the uploaded repository under `/kaggle/input` or clones it from GitHub.
2. Installs the project dependencies.
3. Generates the current reduced benchmark (50 instances per condition, about 1,150 records).
4. Evaluates the five core models in separate processes with auditable step prompts.
5. Creates per-query JSONL and CSV audits, metrics, Markdown reports, plots, and one ZIP archive.

The Hugging Face token must be available as the Kaggle secret/environment variable `HF_TOKEN` for gated models such as Llama.

In [ ]:
# Install Kaggle-side helpers before locating the uploaded project.
%pip install -q bitsandbytes pandas matplotlib
print("Kaggle helper packages installed.")

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Kaggle datasets are mounted read-only under /kaggle/input.
# Upload the repository as a Kaggle Dataset, or set REPO_URL to a public/private clone URL.
REPO_URL = ""
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
PROJECT_DIR = WORK_ROOT / "StateMachine"

if REPO_URL:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    candidates = [
        path for path in KAGGLE_INPUT_ROOT.rglob("run_eval.py")
        if (path.parent / "generate_all.py").exists()
    ]
    if not candidates:
        raise FileNotFoundError(
            "Upload the StateMachine repository as a Kaggle Dataset, or set REPO_URL."
        )
    source_dir = candidates[0].parent
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(source_dir, PROJECT_DIR, ignore=shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc", "data/*.jsonl", "results"
    ))

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project: {PROJECT_DIR}")
print(f"Source: {source_dir if not REPO_URL else REPO_URL}")

In [ ]:
# Install the exact project dependencies after the repository is available.
%pip install -q -r requirements.txt bitsandbytes pandas matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError("A Kaggle GPU runtime is required for the full model evaluation.")

## Kaggle configuration

Keep the repository under `/kaggle/working` because `/kaggle/input` is read-only. Set the `HF_TOKEN` Kaggle secret before running the model cells if you will evaluate Llama. The notebook requests explicit `Step N: <container>` lines, so step-wise scoring measures auditable state answers rather than hidden chain-of-thought.

In [ ]:
import os
from pathlib import Path

# Prefer Kaggle Secrets Manager; never print the token.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret("HF_TOKEN")
        if secret_token:
            os.environ["HF_TOKEN"] = secret_token
    except Exception:
        pass

DATASET = "full"
PRECISION = "4bit"       # Suitable starting point for Kaggle T4/P100 GPUs.
BATCH_SIZE = 1            # Increase only after checking GPU memory.
DEVICE = "auto"
USE_COT = True            # Requests parseable Step N lines for step-wise audit.
OUTPUT_DIR = Path("/kaggle/working/dws_results")
MODELS = [
    "qwen2.5-0.5b",
    "qwen2.5-3b",
    "qwen2.5-7b",
    "llama-3.2-3b",
    "olmo-2-1b",
]

if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is not set. Public models can run; Llama may fail authentication.")
else:
    print("HF_TOKEN detected without printing its value.")

print("Models:", ", ".join(MODELS))
print("Output directory:", OUTPUT_DIR)

In [ ]:
# The old generated JSONL files were removed; regenerate the current reduced benchmark.
dataset_path = PROJECT_DIR / "data" / "full_benchmark.jsonl"
if not dataset_path.exists():
    subprocess.run([sys.executable, "generate_all.py"], cwd=PROJECT_DIR, check=True)
else:
    print(f"Using existing dataset: {dataset_path}")

records = [line for line in dataset_path.open(encoding="utf-8") if line.strip()]
record_count = len(records)
print(f"Full benchmark records: {record_count}")
if record_count < 1000 or record_count > 1200:
    raise ValueError(f"Expected the reduced benchmark to contain about 1,150 records; found {record_count}.")

## Run all core models

Each model is launched in a separate subprocess. This releases model weights before the next model and allows the notebook to continue if one model fails. Each run writes predictions JSONL, an audit CSV containing prompt/response/expected answer/step comparison, metrics JSON, and a Markdown report.

In [ ]:
import time

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_status = {}
for model_name in MODELS:
    print(f"\n{'=' * 80}\nRunning {model_name}\n{'=' * 80}")
    command = [
        sys.executable, "run_eval.py",
        "--model", model_name,
        "--dataset", DATASET,
        "--device", DEVICE,
        "--precision", PRECISION,
        "--batch-size", str(BATCH_SIZE),
        "--output-dir", str(OUTPUT_DIR),
    ]
    if USE_COT:
        command.append("--cot")
    started = time.time()
    completed = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        env=os.environ.copy(),
    )
    run_status[model_name] = {
        "return_code": completed.returncode,
        "elapsed_minutes": round((time.time() - started) / 60, 2),
    }
    if completed.returncode != 0:
        print(f"{model_name} failed; continuing with the remaining models.")

print(run_status)

In [ ]:
# Display final and step-wise accuracy plus audit coverage.
import json
import pandas as pd

rows = []
for model_name in MODELS:
    metrics_file = OUTPUT_DIR / model_name / "full_benchmark_metrics.json"
    if metrics_file.exists():
        metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
        rows.append({
            "model": model_name,
            "instances": metrics["total_instances"],
            "final_accuracy": metrics["overall_accuracy"],
            "stepwise_accuracy": metrics.get("stepwise_accuracy"),
            "stepwise_coverage": metrics.get("stepwise_coverage"),
            "missing_predictions": metrics.get("missing_predictions", 0),
            "runtime_minutes": metrics["elapsed_seconds"] / 60,
        })
summary = pd.DataFrame(rows)
if summary.empty:
    print("No metrics files were found. Check the run status above.")
else:
    display(summary.sort_values("final_accuracy", ascending=False).style.format({
        "final_accuracy": "{:.2%}",
        "stepwise_accuracy": "{:.2%}",
        "stepwise_coverage": "{:.2%}",
        "runtime_minutes": "{:.1f}",
    }))
    summary.to_csv(OUTPUT_DIR / "full_benchmark_summary.csv", index=False)
    print(f"Saved summary: {OUTPUT_DIR / 'full_benchmark_summary.csv'}")

In [ ]:
# Generate publication-style plots and package every artifact for Kaggle output.
plot_command = [
    sys.executable,
    "analysis/plot_results.py",
    "--input", str(OUTPUT_DIR),
    "--output", str(OUTPUT_DIR / "plots"),
]
subprocess.run(plot_command, cwd=PROJECT_DIR, check=True)

archive_path = shutil.make_archive(
    str(WORK_ROOT / "dws_bench_kaggle_results"),
    "zip",
    root_dir=OUTPUT_DIR.parent,
    base_dir=OUTPUT_DIR.name,
)
print(f"Created archive: {archive_path}")
print("Kaggle output directory:", OUTPUT_DIR)
print("Download the ZIP from the notebook output/files panel after the run completes.")